In [32]:
#complete all imports

import pandas as pd
import numpy as np
import pymysql
from sqlalchemy import create_engine, text
import getpass
from dotenv import load_dotenv
import os


In [33]:
#load password

load_dotenv()
password = os.getenv("mysql")

In [34]:
## building a connection 

user = "root"
host = "localhost"
database = "sakila"

In [ ]:
### Q1: Connection and Chel

In [35]:
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}/{database}")
engine.connect()

In [36]:
pd.read_sql("show tables from sakila;", engine)

,Tables_in_sakila
0,actor
1,actor_info
2,address
3,category
4,city
5,country
6,customer
7,customer_list
8,customer_rental_summary
9,film


In [20]:
pd.read_sql(" select * from rental;", engine)

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53
...,...,...,...,...,...,...,...
16039,16045,2005-08-23 22:25:26,772,14,2005-08-25 23:54:26,1,2006-02-15 21:30:53
16040,16046,2005-08-23 22:26:47,4364,74,2005-08-27 18:02:47,2,2006-02-15 21:30:53
16041,16047,2005-08-23 22:42:48,2088,114,2005-08-25 02:48:48,2,2006-02-15 21:30:53
16042,16048,2005-08-23 22:43:07,2019,103,2005-08-31 21:33:07,1,2006-02-15 21:30:53


+++ Question 2

In [ ]:
#building SQL logic
"""


select *
from rental 
where MONTH (rental_date) = :month
    AND
     YEAR (rental_date) = :year

"""

In [21]:


def rentals_month(engine, month, year):
    """
    Retrieve rental data for a given month and year from the rental table.
    Returns a pandas DataFrame (empty if no data).
    """
    
    query = """
    SELECT *
    FROM rental
    WHERE MONTH(rental_date) = %(month)s
      AND YEAR(rental_date) = %(year)s;
    """
    
    df = pd.read_sql(
        query,
        con=engine,
        params={"month": month, "year": year}
    )
    
    return df


In [22]:
df = rentals_month(engine, 5, 2005)
df.head()


,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


In [ ]:
### Same as aove but with more info on the names of titles 

"""
SELECT
    r.*,
    i.film_id,
    f.title,
    f.release_year,
    f.rental_rate
FROM rental r
LEFT JOIN inventory i
    ON r.inventory_id = i.inventory_id
LEFT JOIN film f
    ON i.film_id = f.film_id
WHERE MONTH(r.rental_date) = :month
  AND YEAR(r.rental_date) = :year;



"""

In [23]:
def rentals_month_with_film(engine, month, year):
    """
    Retrieve rental data for a given month and year,
    joined with inventory and film tables.
    Returns a pandas DataFrame.
    """
    
    query = """
    SELECT
        r.*,
        i.film_id,
        f.title,
        f.release_year,
        f.rental_rate
    FROM rental r
    LEFT JOIN inventory i
        ON r.inventory_id = i.inventory_id
    LEFT JOIN film f
        ON i.film_id = f.film_id
    WHERE MONTH(r.rental_date) = %(month)s
      AND YEAR(r.rental_date) = %(year)s;
    """
    
    df = pd.read_sql(
        query,
        con=engine,
        params={"month": month, "year": year}
    )
    
    return df


In [24]:
df_joined = rentals_month_with_film(engine, 5, 2005)
df_joined.head()


,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update,film_id,title,release_year,rental_rate
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53,80,BLANKET BEVERLY,2006,2.99
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53,333,FREAKY POCUS,2006,2.99
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53,373,GRADUATE LORD,2006,2.99
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53,535,LOVE SUICIDES,2006,0.99
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53,450,IDOLS SNATCHERS,2006,2.99


+++ Q3

In [25]:
def rental_count_month(df_rentals, month, year):
    """
    Returns a DataFrame with the number of rentals per customer
    for a given month and year.
    """

    # Create dynamic column name
    column_name = f"rentals_{month:02d}_{year}"

    # Handle empty input DataFrame
    if df_rentals.empty:
        return pd.DataFrame(columns=["customer_id", column_name])

    # Group and count rentals
    df_counts = (
        df_rentals
        .groupby("customer_id")["rental_id"]
        .count()
        .reset_index()
        .rename(columns={"rental_id": column_name})
    )

    return df_counts


In [26]:
### calling the above function 

month = 5
year = 2005

df_rentals = rentals_month(engine, month, year)
df_rental_counts = rental_count_month(df_rentals, month, year)

df_rental_counts.head()


,customer_id,rentals_05_2005
0,1,2
1,2,1
2,3,2
3,5,3
4,6,3


In [27]:
df_rentals

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53
...,...,...,...,...,...,...,...
1151,1153,2005-05-31 21:36:44,2725,506,2005-06-10 01:26:44,2,2006-02-15 21:30:53
1152,1154,2005-05-31 21:42:09,2732,59,2005-06-08 16:40:09,1,2006-02-15 21:30:53
1153,1155,2005-05-31 22:17:11,2048,251,2005-06-04 20:27:11,2,2006-02-15 21:30:53
1154,1156,2005-05-31 22:37:34,460,106,2005-06-01 23:02:34,2,2006-02-15 21:30:53


In [ ]:
### Q4:



In [28]:
def compare_rentals(df1, df2):
    """
    Compares rental counts between two months and returns
    a DataFrame with a difference column.
    """

    # Identify rental columns (excluding customer_id)
    col1 = [col for col in df1.columns if col != "customer_id"][0]
    col2 = [col for col in df2.columns if col != "customer_id"][0]

    # Merge on customer_id
    df_combined = pd.merge(
        df1,
        df2,
        on="customer_id",
        how="outer"
    )

    # Replace NaN with 0 for calculations
    df_combined[[col1, col2]] = df_combined[[col1, col2]].fillna(0)

    # Calculate difference
    df_combined["difference"] = df_combined[col1] - df_combined[col2]

    return df_combined
